# N01 张量与自动微分

> **对应理论篇**：[`01-神经网络与反向传播.md`](../../01-基础/01-神经网络与反向传播.md)、[`02-自动微分与优化器.md`](../../01-基础/02-自动微分与优化器.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/optim.py:23`（`adamw_step_fused` 消费 `.grad`）、`common.py:38`（`_detect_compute_dtype` 决定张量 dtype）
> **Colab**：CPU 运行时，全部运行约 30 秒。

## 学习目标
1. 分清张量的 shape / dtype / device 如何决定算子行为；
2. 理解 autograd 的 `requires_grad` / `grad_fn` / `backward` 与梯度累积；
3. 手写小自动微分，与 PyTorch 对照验证链式法则。


In [1]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

## 理论回顾
对标量 $L=f(\mathbf{x})$，反向传播沿计算图应用链式法则。中间节点 $u$ 的伴随 $\bar u=\partial L/\partial u$，
对 $u=g(v)$：$\bar v=\bar u\,\partial g/\partial v$；多消费者时跨路径求和。

$$ \bar v=\sum_{u\in\mathrm{children}(v)}\bar u\,\frac{\partial u}{\partial v} $$


In [2]:
import torch
x = torch.arange(12).reshape(3, 4)
print("shape", x.shape, "| dtype", x.dtype, "| device", x.device)
print("(3,4)+(4,) 广播 ->", (x + torch.ones(4)).shape)
try:
    x @ torch.ones(3, 3)
except RuntimeError as e:
    print("matmul 报错:", str(e)[:70])
print("fp16(2049) =", torch.tensor(2049.0, dtype=torch.float16).item())  # 精度损失


shape torch.Size([3, 4]) | dtype torch.int64 | device cpu
(3,4)+(4,) 广播 -> torch.Size([3, 4])
matmul 报错: mat1 and mat2 shapes cannot be multiplied (3x4 and 3x3)
fp16(2049) = 2048.0


## 实验：autograd 与梯度累积

In [3]:
import torch
x = torch.tensor([2.0, 3.0], requires_grad=True)
z = (x ** 2 * torch.tensor([1.0, 10.0])).sum()
z.backward(retain_graph=True); print("x.grad =", x.grad)          # 2x*[1,10] = [4,60]
z.backward(); print("再 backward =", x.grad)      # 累加成 [8,120]
x.grad.zero_(); z2 = (x ** 2 * torch.tensor([1.0, 10.0])).sum(); z2.backward(); print("清零后 =", x.grad)
w = torch.tensor([5.0], requires_grad=True); v = (w * 2).detach()
print("detach requires_grad:", v.requires_grad)


x.grad = tensor([ 4., 60.])
再 backward = tensor([  8., 120.])
清零后 = tensor([ 4., 60.])
detach requires_grad: False


## 实验：手写自动微分（链式法则）
对 $f=\sin(ab)+e^{a}$，用反拓扑序求梯度，应与 PyTorch 一致。

In [4]:
import math, torch

class Node:
    def __init__(self, val, parents=(), lg=()):
        self.val = float(val); self.parents = parents; self.lg = lg
    def __add__(self, o):
        o = o if isinstance(o, Node) else Node(o)
        return Node(self.val + o.val, (self, o), (1.0, 1.0))
    def __mul__(self, o):
        o = o if isinstance(o, Node) else Node(o)
        return Node(self.val * o.val, (self, o), (o.val, self.val))
    def sin(self): return Node(math.sin(self.val), (self,), (math.cos(self.val),))
    def exp(self):
        e = math.exp(self.val); return Node(e, (self,), (e,))
    def backward(self):
        topo, seen = [], set()
        def build(n):
            if n not in seen:
                seen.add(n)
                for p in n.parents: build(p)
                topo.append(n)
        build(self); g = {id(n): 0.0 for n in topo}; g[id(self)] = 1.0
        for n in reversed(topo):
            for p, lg in zip(n.parents, n.lg): g[id(p)] += g[id(n)] * lg
        return self, g

a, b = Node(1.3), Node(-0.7)
out, g = ((a * b).sin() + a.exp()).backward()
ta = torch.tensor(1.3, requires_grad=True); tb = torch.tensor(-0.7, requires_grad=True)
f = torch.sin(ta * tb) + torch.exp(ta); f.backward()
print(f"手写 df/da={g[id(a)]:.6f} df/db={g[id(b)]:.6f}")
print(f"torch df/da={ta.grad.item():.6f} df/db={tb.grad.item():.6f}")


手写 df/da=3.239675 df/db=0.797869
torch df/da=3.239675 df/db=0.797869


## 思考题
1. 为什么 `backward()` 默认累加梯度？这对梯度累积有何意义？
2. `detach()` 与 `torch.no_grad()` 有何区别？
3. `argmax` / `round` 的梯度是什么？为什么它们常用于「不参与梯度」的分支？

## 改动实验（任选 2–3 个）
1. 把 $L=x^2+3x$ 换成 $L=\sin x+e^x$，对比 autograd 与解析梯度。
2. 把 `float32` 换成 `float16`，观察梯度精度损失。
3. 用 `create_graph=True` 求二阶导。

## 预期输出
```
x.grad = tensor([ 4., 60.])
再 backward = tensor([ 8., 120.])
清零后 = tensor([ 4., 60.])
手写 df/da=... torch df/da=...
```
